# Project 5: Governed Text-to-SQL Analyst

Compare one-shot SQL with schema retrieval, typed planning, SQLGlot AST policy,
read-only DuckDB execution, bounded repair, PII controls, and export approval.
All data is synthetic and final figures must come from saved measured output.

Upgrade v1: both execution paths use tenant-filtered, PII-free snapshots. Earlier
outputs describe the historical implementation, not the upgraded comparison.
Final evaluation reports strict result hashes separately from alias-insensitive
value equivalence on the fixed fixture. Gold SQL never feeds planner repair.

In [ ]:
from pathlib import Path
import os, runpy, sys
if sys.prefix == sys.base_prefix:
    raise RuntimeError("Use the isolated notebook kernel from SECURITY_UPGRADE.md; shared Colab packages are not modified")
candidates = [Path.cwd(), Path.cwd()/"project5", Path("/content/ai_agentic_attemptings/project5")]
PROJECT_ROOT = next((p.resolve() for p in candidates if (p/"src/project5_agent").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the cloned project directory")
check = runpy.run_path(str(PROJECT_ROOT.parent/"tools/notebook_environment.py"))["validate"]
check(PROJECT_ROOT.parent/"requirements-notebook-cpu.hashes.txt")
os.chdir(PROJECT_ROOT)
source_root = PROJECT_ROOT/"src"
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))
print("Isolated Project 5 CPU/API environment verified; GPU mode requires a separately approved GPU lock")

In [3]:
import getpass, os, sys
from project5_agent.config import load_config
EVAL_BACKEND = "local_gpu"  # deterministic | openai | local_gpu
RUN_FULL_EVAL = True
RUN_API_EVAL = EVAL_BACKEND == "openai"
RUN_LOCAL_GPU_EVAL = EVAL_BACKEND == "local_gpu"
config = load_config(PROJECT_ROOT/"config/default.json")
if RUN_API_EVAL and not os.getenv("OPENAI_API_KEY"):
    if "google.colab" in sys.modules:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
    else: key = getpass.getpass("OPENAI_API_KEY (hidden): ")
    if not key: raise RuntimeError("OPENAI_API_KEY is required for API mode")
    os.environ["OPENAI_API_KEY"] = key
if RUN_LOCAL_GPU_EVAL:
    import torch
    if not torch.cuda.is_available(): raise RuntimeError("Select a Colab GPU runtime for local_gpu mode")
config = config.model_copy(update={"planner_mode": EVAL_BACKEND})
print(config.model_dump())

{'project_id': 'project5', 'seed': 20260802, 'planner_mode': 'local_gpu', 'model': 'gpt-5.6-luna', 'reasoning_effort': 'low', 'development_case_count': 10, 'test_case_count': 40, 'max_repair_attempts': 2, 'max_result_rows': 100, 'authorized_regions': ['NA'], 'max_model_calls': 180, 'max_output_tokens': 500, 'max_retries': 2, 'max_estimated_cost_usd': 6.0, 'input_price_per_million_usd': 1.0, 'output_price_per_million_usd': 6.0, 'local_model': 'Qwen/Qwen2.5-Coder-7B-Instruct', 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'local_device': 'cuda', 'local_max_new_tokens': 500, 'schema_top_k': 5}


In [4]:
import subprocess, sys
database_path = PROJECT_ROOT/"data/cache/project5_ecommerce.duckdb"
benchmark_path = PROJECT_ROOT/"data/cache/project5_questions.json"
subprocess.run([sys.executable, "tools/generate_dataset.py"], check=True)
from project5_agent.dataset import load_benchmark
cases = load_benchmark(benchmark_path)
print({"questions": len(cases), "database": database_path.relative_to(PROJECT_ROOT).as_posix()})

{'questions': 50, 'database': 'data/cache/project5_ecommerce.duckdb'}


In [5]:
development_cases = [case for case in cases if case.split == "development"]
test_cases = [case for case in cases if case.split == "test"]
assert (len(development_cases), len(test_cases)) == (10, 40)
assert "gold_sql" not in development_cases[0].public_view()
print({"development": 10, "test": 40})

{'development': 10, 'test': 40}


In [6]:
from project5_agent.analyst import run_baseline
case = next(case for case in development_cases if case.first_attempt_sql)
print(run_baseline(case, database_path).model_dump())

{'question_id': 'Q5-001', 'system': 'baseline', 'status': 'completed', 'sql': "SELECT ROUND(SUM(o.total_amount), 2) AS revenue FROM orders o JOIN customers c ON c.customer_id=o.customer_id WHERE c.region='LATAM'", 'executed': True, 'policy_allowed': None, 'repaired': False, 'attempts': 1, 'result_hash': '7d59c1361c2559bd06769f9cf2c78c0bf1afd0bf1e72465330ea8b69030f0c30', 'result_correct': True, 'pii_leaked': False, 'latency_ms': 25.168875999952434, 'error': None}


In [ ]:
from project5_agent.analyst import DeterministicSQLPlanner, SCHEMA_CATALOG, run_governed
from project5_agent.local_models import EmbeddingSchemaRetriever
from project5_agent.execution import describe_authorized_schema
schema_catalog = describe_authorized_schema(database_path, config.authorized_regions)
repair_case = next(case for case in cases if case.category == "repair")
result = run_governed(repair_case, database_path, config, DeterministicSQLPlanner())
assert result.repaired and result.result_correct
schema_retriever = None
if RUN_LOCAL_GPU_EVAL:
    schema_retriever = EmbeddingSchemaRetriever(schema_catalog, config.embedding_model, config.local_device)
    print({"dense_schema_matches": schema_retriever.retrieve(repair_case.question, config.schema_top_k)})
print(result.model_dump())

In [8]:
import os, subprocess, sys
pytest_env = {**os.environ, "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1"}
try:
    result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], text=True, capture_output=True, env=pytest_env, timeout=120)
except subprocess.TimeoutExpired as exc:
    raise RuntimeError("Project 5 tests exceeded the 120-second Colab limit") from exc
print(result.stdout)
if result.returncode: print(result.stderr); raise RuntimeError("Project 5 tests failed")

..........                                                               [100%]
10 passed in 10.53s



In [ ]:
from runpy import run_path
RunManifest = run_path(str(PROJECT_ROOT.parent / 'tools/run_manifest.py'))['RunManifest']
from project5_agent.analyst import DeterministicSQLPlanner, LocalSQLPlanner, OpenAISQLPlanner
from project5_agent.evaluation import evaluate_project5
from datetime import datetime, timezone
import json
if not RUN_FULL_EVAL:
    print("Set RUN_FULL_EVAL=True in P05-C02 after deterministic tests pass.")
else:
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    result_dir = PROJECT_ROOT/"output/runs/upgrade-v2"/EVAL_BACKEND/run_id
    with RunManifest(PROJECT_ROOT, result_dir, config.model_dump()) as run_manifest:
        (result_dir/"run_config.json").write_text(json.dumps(config.model_dump(), indent=2))
        if RUN_LOCAL_GPU_EVAL:
            source_planner = LocalSQLPlanner(config, retriever=schema_retriever, schema_catalog=schema_catalog)
        elif RUN_API_EVAL:
            source_planner = OpenAISQLPlanner(config, schema_catalog=schema_catalog)
        else:
            source_planner = DeterministicSQLPlanner()
        summary = evaluate_project5(cases, database_path, config, result_dir, source_planner)
        print(summary)
        run_manifest.record(summary, {name: getattr(globals().get(name), 'model_revision', None) for name in ('source_planner', 'local_backend', 'embedding_retriever')})

In [ ]:
import json
if "result_dir" not in globals(): raise RuntimeError("Run P05-C08 first; historical runs are not reused")
path = result_dir/"project5_representative_samples.json"
print(json.loads(path.read_text()) if path.exists() else "Run P05-C08 first.")

In [ ]:
import subprocess, sys
if "result_dir" not in globals(): raise RuntimeError("Run P05-C08 first; historical runs are not reused")
summary_path = result_dir/"project5_final_summary.json"
if summary_path.exists():
    subprocess.run([sys.executable, "tools/generate_report.py", "--summary", str(summary_path), "--output", str(result_dir/"project5_report.pdf")], check=True)
    subprocess.run([sys.executable, "tools/validate_project.py"] + ([] if RUN_LOCAL_GPU_EVAL else ["--require-results"]), check=True)
else: print("Measured summary absent; report generation skipped.")

In [38]:
PROJECT = 5
BUNDLE_SHA256 = 'cb2cd2b59f6cc5d5805461a120e0ca4e0a320898d383f8ef3debb32120e27594'
BUNDLE = '...'
from datetime import datetime, timezone
import base64
import gzip
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys

from google.colab import drive

mount = Path("/content/drive")
if not mount.is_mount():
    drive.mount(str(mount), timeout_ms=180000)
if not mount.is_mount():
    raise RuntimeError("Connect this notebook to the mounted MyPro runtime")
gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
if "A100" not in gpu:
    raise RuntimeError(f"Expected the A100, found {gpu}")
if globals().get("inference_worker") is not None and inference_worker.poll() is None:
    raise RuntimeError("This notebook already has an active evaluation worker")
if subprocess.check_output(
    ["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"], text=True).strip():
    raise RuntimeError("Another process is using the GPU")

runtime = Path("/content/portfolio_inference_v2")
bundle_dir = runtime / f"project{PROJECT}" / BUNDLE_SHA256
tools_dir = bundle_dir / "tools"
tools_dir.mkdir(parents=True, exist_ok=True)
bundle = gzip.decompress(base64.b64decode(BUNDLE))
if hashlib.sha256(bundle).hexdigest() != BUNDLE_SHA256:
    raise RuntimeError("Notebook package hash mismatch")
for name, contents in json.loads(bundle).items():
    destination = (tools_dir / name).resolve()
    if not destination.is_relative_to(tools_dir):
        raise ValueError("Unexpected path in notebook package")
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(contents)

bootstrap = runtime / "bootstrap"
subprocess.run([sys.executable, "-m", "pip", "install", "--target", str(bootstrap),
                "--no-deps", "uv==0.12.17"], check=True, timeout=180)
uv_cache = Path("/content/project1_training_v2/uv-cache")
if not uv_cache.is_dir():
    uv_cache = runtime / "uv-cache"
uv_env = dict(os.environ, PYTHONPATH=str(bootstrap), UV_CACHE_DIR=str(uv_cache),
              UV_HTTP_RETRIES="1")

def uv(*arguments):
    subprocess.run([sys.executable, "-m", "uv", *map(str, arguments)], env=uv_env,
                   check=True, timeout=600)

venv = runtime / "venv"
python = venv / "bin/python"
lock = tools_dir / "requirements-inference-v2.hashes.txt"
fingerprint = hashlib.sha256(lock.read_bytes()).hexdigest()
marker = venv / ".portfolio-lock"
if not python.is_file():
    uv("venv", "--python", "3.11.17", venv)
if not marker.is_file() or marker.read_text() != fingerprint:
    uv("pip", "install", "--python", python, "--require-hashes", "-r", lock)
    uv("pip", "check", "--python", python)
    marker.write_text(fingerprint)

project_root = mount / f"MyDrive/projects/MyProject{PROJECT}"
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_parent = project_root / "outputs/runs/upgrade-v2"
run_parent.mkdir(parents=True, exist_ok=True)
run_dir = run_parent / run_id
inference_log = run_parent / f"{run_id}.log"
inference_env = {
    k: v for k, v in os.environ.items()
    if not any(word in k.upper() for word in ("TOKEN", "SECRET", "PASSWORD", "API_KEY"))
}
inference_env.update(
    HF_HUB_DISABLE_IMPLICIT_TOKEN="1", HF_HUB_DISABLE_XET="1",
    HF_HUB_DISABLE_TELEMETRY="1", PYTHONUNBUFFERED="1", OTEL_SDK_DISABLED="true",
)
with inference_log.open("x") as log:
    inference_worker = subprocess.Popen(
        [str(python), str(tools_dir / "run_inference_cells_v2.py"), "--project",
         str(PROJECT), "--cells", str(tools_dir / "cells.json"), "--output", str(run_dir)],
        env=inference_env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
    )
print({"project": PROJECT, "run_id": run_id, "pid": inference_worker.pid})


{'project': 5, 'run_id': '20261118T100123095228Z', 'pid': 111019}


In [43]:
import json
import subprocess

manifest_path = run_dir / "run_manifest.json"
if manifest_path.is_file():
    print(json.dumps(json.loads(manifest_path.read_text()), indent=2))
exit_code = inference_worker.poll()
print("Worker exit:", exit_code)
if exit_code is None:
    gpu = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"], text=True)
    print("GPU utilization, memory in use:", gpu.strip())
elif exit_code != 0:
    with inference_log.open("rb") as log:
        log.seek(max(0, inference_log.stat().st_size - 4096))
        print(log.read().decode(errors="replace"))
elif PROJECT == 5:
    keys = ("value_result_accuracy_pct", "value_result_cases", "pii_leak_rate_pct",
            "planning_error_pct", "execution_or_repair_error_pct", "end_to_end_p50_ms")
    for arm in ("baseline", "candidate"):
        summary = json.loads((run_dir / arm / "project5_final_summary.json").read_text())
        metrics = {system: {key: summary[system][key] for key in keys}
                   for system in ("baseline", "governed")}
        payload = {"model": summary["config"]["local_model"], "cases": summary["test_questions"],
                   "usage": summary["planner_usage"], "metrics": metrics}
        print(f"SQL model {arm}:", json.dumps(payload))
elif PROJECT == 8:
    summary = json.loads((run_dir / "longmemeval/results/summary.json").read_text())
    print("LongMemEval:", json.dumps(summary))


{
  "project": 5,
  "run_id": "20261118T100123095228Z",
  "status": "complete",
  "gpu": "NVIDIA A100-SXM4-40GB",
  "python": "3.11.17",
  "lock_sha256": "afc112d1c55172cee298cc26e4e3119e2e9d8506dda21494050212c69ed3c906",
  "cell_bundle_sha256": "288b619b79950ac321efb3cf11d88c32444d4885911963eabdf71823bf1a258b",
  "completed_cells": [
    3,
    4,
    5,
    6,
    7,
    9
  ],
  "model_runs": [
    {
      "arm": "baseline",
      "completed_cells": [
        3,
        4,
        5,
        6,
        7,
        9
      ]
    },
    {
      "arm": "candidate",
      "completed_cells": [
        3,
        4,
        5,
        6,
        7,
        9
      ]
    }
  ],
  "active_model": "candidate",
  "active_cell": 9,
  "matched_holdout_sha256": "19f4e251bdc489b731cd1c0e0547625cb632427340da43c15f46c902e54a79f4",
  "seconds": 910.6551661269987,
  "peak_gpu_allocated_bytes": 15425827328
}
Worker exit: 0
SQL model baseline: {"model": "Qwen/Qwen2.5-Coder-7B-Instruct", "cases": 40, "us